# Etapa 5o — ¿La mejora justifica el tiempo? Versión 2 completa contra liviana

**Ronda de mejoras para la entrega final (desvío 14.11 del plan).** Solo desarrollo, la misma validación cruzada por personas. El conjunto de prueba no se usa.

**Motivo.** La versión 2 tarda unos 30 segundos por foto, contra 20 de la versión 1. Medido en una foto (`05n`): el relleno de exudados lleva 2,8 s, la apertura por reconstrucción 4 % 5,3 s y B-COSFIRE 8,1 s, más de la mitad del tiempo. En sus rondas aportaron poco: Matthews +0,0013 (`05i`), +0,001 (`05h`) y +0,0002 con vasos finos +0,2 puntos (`05l`). Las reglas de adopción pedían que el intervalo excluyera el cero, pero no miraban el costo. Esos aportes se midieron por separado y en contextos distintos, así que no se pueden sumar: hay que medir las versiones enteras.

**Candidatas** (todas con Wiener con ruido estimado, división del fondo, entrenamiento equilibrado por grosor, regresión con L1 con λ elegido dentro del entrenamiento y valor fijo):
- **A.** Versión 2 completa: con relleno de exudados, L1 sobre 31 mapas (los 26 de la versión 1, las 2 aperturas por reconstrucción y los 3 B-COSFIRE).
- **C.** Sin B-COSFIRE: con relleno, L1 sobre 28 mapas.
- **B.** Liviana: sin relleno, L1 sobre los 26 mapas de la versión 1 (sin aperturas ni B-COSFIRE).

**Medidas**, en las personas de afuera, a resolución original, en el campo de visión de la base: Matthews, vasos finos, Dice de línea central y precisión, **en las fotos tal como están y con ruido gaussiano de desvío 10** (el mismo ruido que en `05k`). **Tiempo por foto**, medido de a una foto por vez (sin otros procesos en paralelo), en 5 fotos de desarrollo, con los mapas que usa cada candidata en la vuelta 0; y el de la versión 1, como referencia.

**Regla de adopción (fijada antes de correr).** Se adopta la más simple (B, después C, después A) que no sea inferior a A en Matthews, vasos finos y Dice de línea central sin ruido agregado, ni en Matthews con ruido 10 (límite inferior > −0,03; la regla de la más simple entre las no inferiores, sección 5.3 del esquema). Sacar componentes ya adoptados por costo es un desvío de criterio, que se declara.

**Control:** A tiene que reproducir el Matthews de B en `05m` (0,7502).

In [ ]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means

pd.set_option("display.width", 220)
ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
FULL = C.SEGMENT_BACKGROUND_MODE                       # Wiener + relleno + división
LIGHT = "división + Wiener con ruido estimado"
F31 = C.SEGMENT_FEATURES
F28 = [n for n in F31 if n not in C.COSFIRE_FEATURES]
F26 = [n for n in F28 if n not in C.LESION_FEATURES]
CANDS = {"A. completa": (FULL, F31), "C. sin B-COSFIRE": (FULL, F28), "B. liviana": (LIGHT, F26)}
ORDER = list(CANDS)
WORKERS = 4
print(FULL, "|", LIGHT, "|", {c: len(f) for c, (_, f) in CANDS.items()})

## Pasada 1: píxeles de entrenamiento (con y sin relleno)

In [ ]:
def pass1(image_id):
    from src.pipeline import training_pixels
    return image_id, {"full": training_pixels(image_id, background=FULL, features=F31),
                      "light": training_pixels(image_id, background=LIGHT, features=F26)}

t = time.time()
P1 = dict(map_images(pass1, ids, workers=WORKERS))
print(f"Pasada 1: {time.time() - t:.0f} s")


def samples_of(c):
    mode, feats = CANDS[c]
    key, base = ("full", F31) if mode == FULL else ("light", F26)
    cols = [base.index(n) for n in feats]
    return {i: {k: (v[:, cols] if k in ("x", "x_thr") else v) for k, v in P1[i][key].items()} for i in ids}

## Ajuste por vuelta (L1, λ dentro del entrenamiento)

In [ ]:
def fit_job(job):
    from src.pipeline import choose_value, fit_weights_l1
    k, c = job
    sc = samples_of(c)
    tr = list(folds.index[folds != k])
    w, lam, _ = fit_weights_l1(sc, tr, {i: int(folds[i]) for i in tr})
    return job, w, choose_value(sc, w, tr), lam

t = time.time()
fit = {}
for job, w, value, lam in map_images(fit_job, [(k, c) for c in ORDER for k in range(5)], workers=WORKERS):
    fit[job] = (w, value, lam)
print(f"Ajustes: {time.time() - t:.0f} s")
pd.DataFrame([{"candidata": c, "vuelta": k, "λ": fit[(k, c)][2], "mapas usados": int(np.count_nonzero(fit[(k, c)][0][1:])),
               "valor fijo": fit[(k, c)][1]} for c in ORDER for k in range(5)]).set_index(["candidata", "vuelta"]).round(5)

## Pasada 2: las personas de afuera, sin ruido y con ruido 10

In [ ]:
def pass2(image_id):
    import numpy as np
    from src.config import WIDTH_LIMITS_PX
    from src.data import load_fov, load_image, load_vessels
    from src.degrade import gaussian_noise
    from src.enhance import upscale
    from src.fov import detect_fov_robust
    from src.metrics import centerline_dice, matthews
    from src.pipeline import feature_scores
    from src.vessels import skeleton_widths
    k = int(folds[image_id])
    clean = load_image(image_id)
    truth, fov = load_vessels(image_id), load_fov(image_id)
    skel, wd = skeleton_widths(truth & fov)
    thin = skel & (wd <= WIDTH_LIMITS_PX[0])
    rows = []
    for s in (0, 10):
        rgb = clean if s == 0 else gaussian_noise(clean, s, C.SEED + 1000 * s + ids.index(image_id))
        auto = detect_fov_robust(rgb)
        for c in ORDER:
            mode, feats = CANDS[c]
            w, value, _ = fit[(k, c)]
            (p,), _, _ = feature_scores(rgb, [w], 2, background=mode, features=feats)
            m = (upscale(p, fov.shape) > value) & auto
            marked = m & fov
            rows.append({"image": image_id, "ruido": s, "candidata": c, "matthews": matthews(m, truth, fov),
                         "finos": m[thin].mean(), "dice_linea_central": centerline_dice(m, truth, fov),
                         "precision": (marked & truth).sum() / max(marked.sum(), 1)})
    return rows

t = time.time()
R = pd.DataFrame([r for rows in map_images(pass2, ids, workers=WORKERS) for r in rows])
R.to_csv(ROOT / "results" / "05o_v2_liviana.csv", index=False)
print(f"Pasada 2: {time.time() - t:.0f} s")

## Resultados

In [ ]:
MEAS = ("matthews", "finos", "dice_linea_central", "precision")
rows = []
for c in ORDER[1:]:
    for s in (0, 10):
        sub = R[R["ruido"] == s]
        for meas in MEAS if s == 0 else ("matthews",):
            w = sub.pivot(index="image", columns="candidata", values=meas)
            d = (w[c] - w[ORDER[0]]).to_frame()
            lo, hi = percentile_interval(person_bootstrap_means(d, dev["person"]))
            rows.append({"candidata": c, "ruido": s, "medida": meas, "A": w[ORDER[0]].mean(), "candidata media": w[c].mean(),
                         "diferencia con A": d.iloc[:, 0].mean(), "inferior": lo[0], "superior": hi[0],
                         "imágenes que mejoran": int((d.iloc[:, 0] > 0).sum())})
table = pd.DataFrame(rows).set_index(["candidata", "ruido", "medida"])
print("Medias:")
display(R.pivot_table(index="candidata", columns="ruido", values=list(MEAS)).loc[ORDER].round(4))
table.round(4)

In [ ]:
def ok(c):
    t = table.loc[c]
    return bool(all(t.loc[(0, m), "inferior"] > -C.NONINF_MARGIN_MATTHEWS for m in ("matthews", "finos", "dice_linea_central"))
                and t.loc[(10, "matthews"), "inferior"] > -C.NONINF_MARGIN_MATTHEWS)

status = {c: ok(c) for c in ("B. liviana", "C. sin B-COSFIRE")}
adopt = next((c for c in ("B. liviana", "C. sin B-COSFIRE") if status[c]), ORDER[0])
print("No inferiores:", status, "→ se adopta:", adopt)

## Tiempo por foto

In [ ]:
from src.data import load_image
from src.enhance import upscale
from src.fov import detect_fov_robust
from src.pipeline import WEIGHTS_PATH, feature_scores, segment_final

timing_ids = ids[::6]
rows = []
for image_id in timing_ids:
    rgb = load_image(image_id)
    t = time.time()
    segment_final(rgb, path=WEIGHTS_PATH)
    rows.append({"imagen": image_id, "versión": "versión 1", "segundos": time.time() - t})
    for c in ORDER:
        mode, feats = CANDS[c]
        w, value, _ = fit[(0, c)]
        used = [n for n, x in zip(feats, w[1:]) if x != 0]
        wu = np.r_[w[0], [x for x in w[1:] if x != 0]]
        t = time.time()
        (p,), _, _ = feature_scores(rgb, [wu], 2, background=mode, features=used)
        m = (upscale(p, rgb.shape[:2]) > value) & detect_fov_robust(rgb)
        rows.append({"imagen": image_id, "versión": c, "segundos": time.time() - t})
T = pd.DataFrame(rows)
print("Tiempo por foto (de a una, sin procesos en paralelo):")
T.groupby("versión")["segundos"].agg(["mean", "min", "max"]).loc[["versión 1"] + ORDER].round(1)

In [ ]:
print("Por grupo (media), sin ruido agregado:")
R[R["ruido"] == 0].join(dev["group"], on="image").pivot_table(index="candidata", columns="group",
                                                               values=["matthews", "finos", "precision"]).loc[ORDER].round(3)

## Decisión (escrita después de correr)

*(pendiente)*